# SOLiD-PINN v0.9.0c — Coupled resistance closure
CPU · Run all · No Drive · No token. Nine numerical cases, five discretizations, 72 fits only if numerical gate passes. Synthetic normalized linearized x-r bridge: global interface state changes spatially varying reaction resistance, not a full electrochemical P2D or collocation PINN. Preserve failed gates and unsuccessful fits in JSON.
## 1. Pinned source and tests


In [ ]:
from pathlib import Path
from google.colab import files
import subprocess,sys,os,tempfile,json,hashlib,math
REV="966e4023927ceb2ad01cbf6a168978f04b9a1e50"
repo=Path(tempfile.mkdtemp(prefix="solid-v090c-"))/"repo"
subprocess.run(["git","clone","--no-checkout","https://github.com/theerayutbo/SOLiD-PINN.git",str(repo)],check=True)
subprocess.run(["git","checkout","--detach",REV],cwd=repo,check=True)
assert subprocess.check_output(["git","rev-parse","HEAD"],cwd=repo,text=True).strip()==REV
hashes={"coupled_bridge.py":"87f474918e731710193a4b01ee2075d5cc6a157e12fc1c9e1fdcf3779b37d3e3","test_coupled_bridge.py":"8d1e7a81d5cb10c77338e4fab2543da146fc103e8a60dfe463df5ed35424d2de","hybrid_p2d.py":"20aaeede052a29a2b65b1cd322b9fade3b4094fd4e1564d15787d7e240c51f18"}
for name,digest in hashes.items():
    assert hashlib.sha256((repo/"scripts"/name).read_bytes()).hexdigest()==digest
env=dict(os.environ,OPENBLAS_NUM_THREADS="1",OMP_NUM_THREADS="1",MKL_NUM_THREADS="1")
subprocess.run([sys.executable,"-m","pip","install","numpy==2.3.5","scipy==1.17.0","pytest"],check=True)
subprocess.run([sys.executable,"-m","pytest","-q","scripts/test_coupled_bridge.py"],cwd=repo,env=env,check=True)


## 2. Numerical gate and missing-physics benchmark
Training 32×48, four substeps per sample. Reference 64×96, 16 substeps. Calibrated, linear, RBF and known-law oracle; three truths, two noise levels, three seeds. Check inventory, algebraic charge, resistive power identity and sampled state bounds. No adaptive gate or real battery data.


In [ ]:
out=repo/"results.json"
subprocess.run([sys.executable,"-u","scripts/coupled_bridge.py","--out",str(out)],cwd=repo,env=env,check=True)
d=json.loads(out.read_text())
truths=("null","rest","peak");profiles=("train","test","extra");methods=("calibrated","linear","rbf","oracle")
assert len(d["numerical_checks"])==9
assert {(x["truth"],x["profile"]) for x in d["numerical_checks"]}=={(t,k) for t in truths for k in profiles}
def healthy(c):
    return c["mass"]<1e-9 and c["charge"]<1e-9 and c["power"]<1e-9 and c["min_dissipation"]>=-1e-12 and 0<c["min_c"]<=c["max_c"]<1 and 0<c["min_surface"]<=c["max_surface"]<1 and 0<=c["min_x"]<=c["max_x"]<=1
for x in d["numerical_checks"]:
    assert len(x["delta_mv"])==5 and len(x["checks"])==5
    assert x["passed"]==(max(x["delta_mv"].values())<=1 and all(healthy(c) for c in x["checks"].values()))
gate=all(x["passed"] for x in d["numerical_checks"])
assert d["numerical_gate_passed"]==gate
if gate:
    assert len(d["metrics"])==72
    assert {(x["truth"],x["noise"],x["seed"],x["method"]) for x in d["metrics"]}=={(t,n,s,m) for t in truths for n in (1.,5.) for s in (11,23,37) for m in methods}
    for x in d["metrics"]:
        assert set(x["checks"])==set(profiles)
        assert x["physical_checks_passed"]==all(healthy(c) for c in x["checks"].values())
        assert all(math.isfinite(x[k]) and x[k]>=0 for k in ("train_rmse_mv","test_rmse_mv","extra_rmse_mv","closure_rmse"))
else:
    assert not d["metrics"]
d.update(source_revision=REV,source_hashes=hashes,targeted_tests_passed=4)
d["fit_gate_passed"]=gate and all(x["success"] and x["physical_checks_passed"] for x in d["metrics"])
print("Numerical gate:",gate,"Fit gate:",d["fit_gate_passed"],"Fits:",len(d["metrics"]))


## 3. Export evidence
Download JSON even if numerical or fit gate fails. Temporary runtime only; download before disconnecting.


In [ ]:
raw=(json.dumps(d,sort_keys=True,indent=2,allow_nan=False)+"\n").encode()
sha=hashlib.sha256(raw).hexdigest()
root=Path("/content/SOLiD-PINN/v090c/handoff");root.mkdir(parents=True,exist_ok=True)
dest=root/f"SOLiD-PINN_v090c_colab_{sha[:20]}.json"
if dest.exists(): assert dest.read_bytes()==raw
else:
    tmp=dest.with_suffix(".part");tmp.write_bytes(raw);tmp.replace(dest)
assert dest.read_bytes()==raw
print("LOCAL FILE VERIFIED",dest.name,sha)
try: files.download(str(dest))
except Exception as e: print("Download manually from Files panel:",str(dest),str(e))
